# Lesson 4.4: Collaborative Location Review


**🎬 Video:** [Lesson 4.4: Collaborative Location Review](#)

## Overview

The geoparser resolved hundreds of place names from Reddit posts, but automatic geocoders make predictable mistakes:

| Error type | Example |
|---|---|
| Wrong disambiguation | `JMU` → Jiamusi, China instead of James Madison University |
| Wrong coordinates | `Harrisonburg` placed in the wrong county |
| False positive | `Virginia` captured as a destination when it is just context |
| Spurious match | `D-Hall` → a mall in New York |

Your team will work through these locations together in Google Sheets, then export a clean file for Lessons 5 and 6.

**Workflow overview**

| Part | Tool | Who |
|---|---|---|
| A — Prepare the review file | Python (this notebook) | One student |
| B — Review locations | Google Sheets | Whole team |
| C — Load and verify from Google Sheets | Python (this notebook) | One student |
| D — Export the cleaned file | Python (this notebook) | One student |
| E — Commit to Git | Codespaces | One student |


---

## 1 Load the Review File

Run the cell below to confirm the file is ready. The six review columns (`action`, `corrected_name`, `corrected_lat`, `corrected_lon`, `corrected_place_type`, `reviewer`) are already included — they were added by the pipeline when the data was generated.


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/JMU/JMU_geoparsed_long.csv")

print(f"✅ Loaded: {len(df):,} rows  |  {len(df.columns)} columns")
print(f"\nplace_type distribution:")
print(df["place_type"].value_counts(dropna=False).to_string())
print(f"\nNext: open  ../data/JMU/JMU_geoparsed_long.csv  in Google Sheets → follow Part B.")


---

## 2 Review in Google Sheets

**One student** starts this step; the whole team contributes.

### B1 — Import the file

1. Go to [sheets.google.com](https://sheets.google.com) → **Blank spreadsheet**
2. **File → Import → Upload** → select `data/JMU/JMU_geoparsed_long.csv` from the repo root
3. Choose **Replace spreadsheet**, separator type **Comma**
4. Rename the spreadsheet: `JMU_geoparsed_long_cleaned`
5. **Share → Anyone with the link → Editor** → copy the link and post it in your team channel

> 💡 **Tip:** Sort by `place_count` descending first — high-frequency places matter most and are worth checking carefully.

### B2 — Add data validation

Set up three dropdown validations so the whole team fills in consistent values.

**Column `action`** *(most important)*:

1. Click the `action` column header to select the whole column
2. **Data → Data validation → Add rule**
3. Criteria: **Dropdown** → add three options: `KEEP`, `CORRECT`, `REMOVE`
4. "If data is invalid": **Reject input**
5. Right-click the `action` header cell → **Insert note** → paste: *KEEP = location is correct. CORRECT = right place, wrong details. REMOVE = not a real location or geoparser error.*

**Column `reviewer`**:

1. Select the `reviewer` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add each team member's name

**Column `corrected_place_type`**:

1. Select the `corrected_place_type` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add: `Country`, `State`, `Region`, `City`, `Neighborhood`, `University`, `Road`, `Building`, `Natural Feature`

### B3 — Review the rows

Work through the rows as a team. For each location:

- Read the `sentences` column to understand the context
- Check `place`, `latitude`, `longitude`, and `place_type`
- Set `action` to `KEEP`, `CORRECT`, or `REMOVE`
- If `CORRECT`: fill in only the columns that need changing:
  - `corrected_name` — new place name
  - `corrected_latlon` — paste directly from Google Maps (e.g. `38.433998, -78.872973`)
  - `corrected_place_type` — select from the dropdown
- Enter your name in `reviewer`

> 💡 **Getting coordinates from Google Maps:** Right-click any spot on the map → click the coordinates at the top of the menu → they copy automatically. Paste the full string (`38.433998, -78.872973`) into `corrected_latlon` — the comma is fine, Python will split it on import.

### B4 — Publish the sheet

Once your team has reviewed all rows:

1. **File → Share → Publish to web**
2. Choose: **Entire document** → **Comma-separated values (.csv)**
3. Click **Publish** → copy the URL
4. Paste the URL into the `SHEETS_LINK` variable in **Part C** below


---

## 3 Load from Google Sheets and Verify

Paste your published sheet URL into `SHEETS_LINK` in the cell below and run both cells. The map lets you visually check for any remaining misplaced pins before you export.


In [ ]:
import sys; sys.path.insert(0, "../tests")
from helpers import load_and_validate_review_sheet

# ──────────────────────────────────────────────────────────────────────────────
# ⚠️  BEFORE RUNNING THIS CELL:
#   1. Complete Part B (review all rows in Google Sheets)
#   2. File → Share → Publish to web → Entire document → CSV → Publish
#   3. Copy the published URL and paste it below, replacing the placeholder
# ──────────────────────────────────────────────────────────────────────────────

SHEETS_LINK = "https://docs.google.com/spreadsheets/d/PASTE_YOUR_LINK_HERE/pub?output=csv"

if "PASTE_YOUR_LINK_HERE" in SHEETS_LINK:
    print("❌ Update SHEETS_LINK before running this cell.")
    print("   File → Share → Publish to web → Entire document → CSV → Publish → copy URL.")
else:
    df_review = load_and_validate_review_sheet(SHEETS_LINK)


In [ ]:
# Verification map — inspect dots geographically
# Points far outside Virginia / the US East Coast are worth checking

if "df_review" not in dir():
    print("❌ df_review is not loaded yet.")
    print("   Run the cell above first (paste your SHEETS_LINK and run it), then re-run this cell.")
else:
    df_map = df_review[
        df_review["action"].isin(["KEEP", "CORRECT"]) |
        df_review["action"].isna() |
        (df_review["action"] == "")
    ].copy()

    df_map["lat_plot"] = pd.to_numeric(
        df_map["corrected_lat"].where(
            df_map["corrected_lat"].notna() & (df_map["corrected_lat"].astype(str).str.strip() != ""),
            df_map["latitude"]
        ), errors="coerce")

    df_map["lon_plot"] = pd.to_numeric(
        df_map["corrected_lon"].where(
            df_map["corrected_lon"].notna() & (df_map["corrected_lon"].astype(str).str.strip() != ""),
            df_map["longitude"]
        ), errors="coerce")

    df_map = df_map.dropna(subset=["lat_plot", "lon_plot"])

    fig = px.scatter_map(
        df_map,
        lat="lat_plot", lon="lon_plot",
        hover_name="place",
        hover_data={
            "place_type": True,
            "action": True,
            "lat_plot": False,
            "lon_plot": False,
        },
        color="action",
        color_discrete_map={
            "KEEP": "#2ca02c",
            "CORRECT": "#ff7f0e",
            "": "#aec7e8",
        },
        size_max=12,
        map_style="carto-positron",
        center={
            "lat": 37.5,
            "lon": -78.0,
        },
        zoom=4,
        height=500,
        title="Location review map — hover for details"
    )
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.show()

    print("\n💡 Dots far outside Virginia may be geoparser errors.")
    print("   Go back to the sheet and mark them REMOVE if needed, then re-run Part C.")

---

## 4 Export the Cleaned File

When your team is satisfied with the review, run the cell below. It will:

- Drop every row marked `REMOVE`
- Apply any corrections (`corrected_name` → `place`, `corrected_lat` → `latitude`, etc.)
- Remove the six review columns
- Save the result as `../data/JMU/JMU_geoparsed_cleaned.csv`

This file feeds directly into Lessons 5 and 6.


In [ ]:
# Apply corrections and export the cleaned file

if "df_review" not in dir():
    print("❌ df_review is not loaded yet.")
    print("   Run the Part C cell first (paste your SHEETS_LINK and run it), then re-run this cell.")
else:
    df_out = df_review.copy()

    long_path  = "../data/JMU/JMU_geoparsed_long.csv"
    clean_path = "../data/JMU/JMU_geoparsed_cleaned.csv"

    # Drop rows marked REMOVE
    n_before = len(df_out)
    df_out = df_out[df_out["action"] != "REMOVE"].copy()
    n_removed = n_before - len(df_out)

    # Apply corrections where non-empty
    def apply_correction(df, corrected_col, target_col, cast=None):
        mask = df[corrected_col].notna() & (df[corrected_col].astype(str).str.strip() != "")
        if cast:
            df.loc[mask, target_col] = pd.to_numeric(df.loc[mask, corrected_col], errors="coerce")
        else:
            df.loc[mask, target_col] = df.loc[mask, corrected_col]
        return df, mask.sum()

    df_out, n_name = apply_correction(df_out, "corrected_name",       "place")
    df_out, n_lat  = apply_correction(df_out, "corrected_lat",        "latitude",  cast=True)
    df_out, n_lon  = apply_correction(df_out, "corrected_lon",        "longitude", cast=True)
    df_out, n_type = apply_correction(df_out, "corrected_place_type", "place_type")

    # ── Write reviewed state back to long CSV (preserves action columns for grading) ──
    # Drop the split helper columns (corrected_lat/lon) — they're derived from corrected_latlon
    keep_cols = [c for c in df_review.columns if c not in ["corrected_lat", "corrected_lon"]]
    df_review[keep_cols].to_csv(long_path, index=False)
    print(f"✅ Updated  {long_path}")

    # ── Save cleaned file ──────────────────────────────────────────────────────────
    review_cols = ["action", "corrected_name", "corrected_latlon", "corrected_lat", "corrected_lon",
                   "corrected_place_type", "reviewer", "place_count"]
    df_out = df_out.drop(columns=[c for c in review_cols if c in df_out.columns])
    df_out.to_csv(clean_path, index=False)

    print(f"✅ Saved    {clean_path}")
    print(f"\nSummary:")
    print(f"  {n_before:,} rows in  →  {len(df_out):,} rows out  ({n_removed} removed)")
    print(f"  Names corrected:        {n_name}")
    print(f"  Coordinates corrected:  {n_lat} lat  /  {n_lon} lon")
    print(f"  Place types corrected:  {n_type}")
   


---

## 5 Commit to Git

**One student on the team** does this step after the export is complete.

This is the same branch → commit → pull request workflow from [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb). Refer back to that lesson if you need a refresher.

1. Create a new branch named `location-review` from the status bar
2. Open **Source Control** and stage `data/JMU/JMU_geoparsed_long.csv` and `data/JMU/JMU_geoparsed_cleaned.csv`
3. Commit with a message like `review: update JMU location data`, then **Commit & Sync**
4. Publish the branch and open a Pull Request from `location-review` → `main`
5. Merge the PR on GitHub, delete the branch, switch back to `main`, and sync

> 👉 **Note:** *Do not commit any other files. If unexpected files appear under Changes, discard them before committing.*


---

## 6 Check Your Progress

Run the cell below at any time to see how many locations your team has reviewed, which checks pass, and your current grade estimate.


In [ ]:
%run ../tests/progress.py


---

## Lesson Summary

### Part 1: Load the Review File
- `pd.read_csv('file.csv')` — loads the raw geoparsed output for inspection before review

### Part 2: Review in Google Sheets
- Export the DataFrame to CSV, import it into Google Sheets, add data-validation dropdowns, and have teammates review each row for accuracy
- The shared sheet is the single source of truth for the team's location decisions — every member should contribute rows

### Part 3: Load from Google Sheets and Verify
- `pd.to_numeric(df['col'], errors='coerce')` — converts a column to numbers, turning any non-numeric values into `NaN` so they can be caught and fixed
- `df['col'].str.strip()` — removes accidental leading/trailing spaces introduced during manual editing

### Part 4: Export the Cleaned File
- `df.dropna(subset=['latitude', 'longitude'])` — removes rows that are missing coordinates after review
- `df.to_csv('file.csv', index=False)` — saves the verified dataset; this file is the input for Lesson 5

### Part 5: Commit to Git and Check Your Progress
- All team members commit the reviewed CSV to the shared repository so everyone works from the same cleaned data in the next lesson

➡️ **Next:** [Lesson 5.1 — Sentiment Analysis](../lesson_5_sentiment_analysis/lesson_5_1_sentiment_analysis.ipynb)